In [158]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import joblib
from src.database import inspect_table_columns

from DataCleaner import DataCleaner


from   sklearn.pipeline  import Pipeline      # Enchaînement des transformations
from   sklearn.compose   import ColumnTransformer # Application par colonnes
from   sklearn.impute    import SimpleImputer # Gestion des valeurs manquantes
from   sklearn.preprocessing import (
    StandardScaler,                           # Normalisation des données
    RobustScaler,
    OneHotEncoder,                            # Encodage catégoriel binaire
    FunctionTransformer                       # Transformations personnalisées
)



In [156]:
PREPROCESED_DIR  = Path("../data/processed")

FEATURE_TARGET               = "target_attrition"

# --- LISTE DE COLUMNAS (Pour selection) ---
FEATURES_TO_REMOVE            = []  # COLS_IDENTIFIANTS + COLS_CONSTANT + COLS_MISSING_EXCESIF + self.COLS_REDONDANTS
COLS_IDENTIFIANTS             = ['id', 'id_employee', 'eval_number', 'code_sondage' ]
COLS_CONSTANT                 = ['nombre_heures_travailless','nombre_employee_sous_responsabilite', 'ayant_enfants']
COLS_MISSING_EXCESIF          = []
COLS_REDONDANTS               = []

# --- CATEGORICAL  ---
COLS_CATEGORICAL              = []  # Todas las categóricas detectadas
COLS_ONE_HOT_ENCODING         = ['genre', 'statut_marital', 'poste', 'domaine_etude', 'frequence_deplacement']
COLS_BINARY_ENCODING          = []
COLS_TARGET_ENCODING          = []
COLS_TARGET_ADVANCED_ENCODING = []

# --- NUMERICAL ---
COLS_NUMERICAL                = []  # Todas las numéricas detectadas
COLS_TO_LOG                   = [ 'revenu_mensuel', 'annee_experience_totale','annees_dans_l_entreprise', 'annees_depuis_la_derniere_promotion']
COLS_TO_ROBUST                = []
COLS_STANDARD                 = ['age',
                                 'nombre_experiences_precedentes',
                                 'annees_dans_le_poste_actuel',
                                 'satisfaction_employee_environnement',
                                 'note_evaluation_precedente',
                                 'satisfaction_employee_nature_travail',
                                 'satisfaction_employee_equipe',
                                 'satisfaction_employee_equilibre_pro_perso',
                                 'note_evaluation_actuelle',
                                 'heure_supplementaires',
                                 'augementation_salaire_precedente',
                                 'nombre_participation_pee',
                                 'nb_formations_suivies',
                                 'distance_domicile_travail',
                                 'niveau_education',
                                 'annes_sous_responsable_actuel',
                                 'attrition_binary']

## 📗 Creation du un META-DICTIONAIRE 

In [136]:
meta_dict_enrichi = {
    # === VARIABLE CIBLE ===
    "a_quitte_l_entreprise"                     : {
        "description"        : "A quitté l'entreprise (Variable cible)",
        "type"               : "object",
        "categorie"          : "Target",
        "unite"              : None,
        "a_supprimer"        : True,
        "raison_suppression" : "VARIABLE CIBLE - Deja codifiqué dans la phase EDA",
        "valeurs_possibles"  : ["Oui", "Non"],
    },

    # === IDENTIFIANTS ===
    "id"                                        : {
        "description"        : "Identifiant secondaire",
        "type"               : "object",
        "categorie"          : "Identifiant",
        "unite"              : None,
        "a_supprimer"        : True,
        "raison_suppression" : "Identifiant utilisé dans le merge"
    },
    "id_employee"                               : {
        "description"        : "Identifiant unique de l'employé",
        "type"               : "int64",
        "categorie"          : "Identifiant",
        "unite"              : None,
        "a_supprimer"        : True,
        "raison_suppression" : "Identifiant technique sans valeur prédictive (df_sirh)"
    },
    "code_sondage"                              : {
        "description"        : "Code du sondage RH",
        "type"               : "int64",
        "categorie"          : "Identifiant",
        "unite"              : None,
        "a_supprimer"        : True,
        "raison_suppression" : "Identifiant de sondage sans valeur prédictive (df_sondage)"
    },
    "eval_number"                               : {
        "description"        : "Numéro ou type d'évaluation",
        "type"               : "object",
        "categorie"          : "Identifiant",
        "unite"              : None,
        "a_supprimer"        : True,
        "raison_suppression" : "Identifiant de sondage sans valeur prédictive (evals)"
    },

    # === CONSTANTS ===

    "nombre_employee_sous_responsabilite"       : {
        "description"        : "Nombre d'employés sous sa responsabilité",
        "type"               : "int64",
        "categorie"          : "Management",
        "unite"              : "nombre",
        "a_supprimer"        : True,
        "raison_suppression" : "Constant"
    },
    "nombre_heures_travailless"                 : {
        "description"        : "Nombre d'heures de travail hebdomadaires",
        "type"               : "int64",
        "categorie"          : "Temps de travail",
        "unite"              : "heures/semaine",
        "a_supprimer"        : True,
        "raison_suppression" : "Constant"
    },

    "ayant_enfants"                             : {
        "description"        : "A des enfants",
        "type"               : "object",
        "categorie"          : "Démographique",
        "unite"              : None,
        "a_supprimer"        : True,
        "raison_suppression" : "Constant"
    },
    # === DONNÉES DÉMOGRAPHIQUES ===
    "age"                                       : {
        "description"        : "Âge de l'employé",
        "type"               : "int64",
        "categorie"          : "Démographique",
        "unite"              : "années",
        "a_supprimer"        : False,
    },
    "genre"                                     : {
        "description"        : "Genre de l'employé",
        "type"               : "object",
        "categorie"          : "Démographique",
        "unite"              : None,
        "a_supprimer"        : False,
    },
    "statut_marital"                            : {
        "description"        : "Statut matrimonial",
        "type"               : "object",
        "categorie"          : "Démographique",
        "unite"              : None,
        "a_supprimer"        : False,
    },

    "distance_domicile_travail"                 : {
        "description"        : "Distance entre domicile et travail",
        "type"               : "int64",
        "categorie"          : "Démographique",
        "unite"              : "km",
        "a_supprimer"        : False,
    },

    # === ÉDUCATION ET FORMATION ===
    "niveau_education"                          : {
        "description"        : "Niveau d'éducation",
        "type"               : "int64",
        "categorie"          : "Éducation",
        "unite"              : "niveau (1-5)",
        "a_supprimer"        : False,
        "encodage_necessaire": False,
        "note"               : "Variable ordinale déjà encodée"
    },
    "domaine_etude"                             : {
        "description"        : "Domaine d'études principal",
        "type"               : "object",
        "categorie"          : "Éducation",
        "unite"              : None,
        "a_supprimer"        : False,
    },
    "nb_formations_suivies"                     : {
        "description"        : "Nombre de formations suivies dans l'entreprise",
        "type"               : "int64",
        "categorie"          : "Éducation",
        "unite"              : "nombre",
        "a_supprimer"        : False,
    },

    # === POSTE ET DÉPARTEMENT ===
    "departement"                               : {
        "description"        : "Département de travail",
        "type"               : "object",
        "categorie"          : "Poste",
        "unite"              : None,
        "a_supprimer"        : False,
    },
    "poste"                                     : {
        "description"        : "Intitulé du poste occupé",
        "type"               : "object",
        "categorie"          : "Poste",
        "unite"              : None,
        "a_supprimer"        : False,
    },
    "niveau_hierarchique_poste"                 : {
        "description"        : "Niveau hiérarchique du poste",
        "type"               : "int64",
        "categorie"          : "Poste",
        "unite"              : "niveau (1-5)",
        "a_supprimer"        : False,
    },

    # === RÉMUNÉRATION ===
    "revenu_mensuel"                            : {
        "description"        : "Revenu mensuel brut",
        "type"               : "int64",
        "categorie"          : "Rémunération",
        "unite"              : "€",
        "a_supprimer"        : False,
    },
    "augementation_salaire_precedente"          : {
        "description"        : "Pourcentage d'augmentation salariale précédente",
        "type"               : "object",
        "categorie"          : "Rémunération",
        "unite"              : "%",
        "a_supprimer"        : False,
    },

    # === EXPÉRIENCE PROFESSIONNELLE ===
    "nombre_experiences_precedentes"            : {
        "description"        : "Nombre d'employeurs précédents",
        "type"               : "int64",
        "categorie"          : "Expérience",
        "unite"              : "nombre",
        "a_supprimer"        : False,
    },
    "annee_experience_totale"                   : {
        "description"        : "Années d'expérience professionnelle totale",
        "type"               : "int64",
        "categorie"          : "Expérience",
        "unite"              : "années",
        "a_supprimer"        : False,
    },
    "annees_dans_l_entreprise"                  : {
        "description"        : "Ancienneté dans l'entreprise",
        "type"               : "int64",
        "categorie"          : "Expérience",
        "unite"              : "années",
        "a_supprimer"        : False,
    },
    "annees_dans_le_poste_actuel"               : {
        "description"        : "Ancienneté dans le poste actuel",
        "type"               : "int64",
        "categorie"          : "Expérience",
        "unite"              : "années",
        "a_supprimer"        : False,
    },
    "annees_depuis_la_derniere_promotion"       : {
        "description"        : "Années écoulées depuis la dernière promotion",
        "type"               : "int64",
        "categorie"          : "Expérience",
        "unite"              : "années",
        "a_supprimer"        : False,
    },

    # === TEMPS DE TRAVAIL ===

    "heure_supplementaires"                     : {
        "description"        : "Fait des heures supplémentaires",
        "type"               : "object",
        "categorie"          : "Temps de travail",
        "unite"              : None,
        "a_supprimer"        : False,
    },
    "frequence_deplacement"                     : {
        "description"        : "Fréquence des déplacements professionnels",
        "type"               : "object",
        "categorie"          : "Temps de travail",
        "unite"              : None,
        "a_supprimer"        : False,
    },

    # === ÉVALUATIONS ET PERFORMANCE ===
    "note_evaluation_precedente"                : {
        "description"        : "Note de l'évaluation précédente",
        "type"               : "int64",
        "categorie"          : "Performance",
        "unite"              : "score (1-100)",
        "a_supprimer"        : False,
        "encodage_necessaire": False
    },
    "note_evaluation_actuelle"                  : {
        "description"        : "Note de l'évaluation actuelle",
        "type"               : "int64",
        "categorie"          : "Performance",
        "unite"              : "score (1-100)",
        "a_supprimer"        : False,
    },


    # === SATISFACTION AU TRAVAIL ===
    "satisfaction_employee_environnement"       : {
        "description"        : "Satisfaction vis-à-vis de l'environnement de travail",
        "type"               : "int64",
        "categorie"          : "Satisfaction",
        "unite"              : "score (1-4)",
        "a_supprimer"        : False,
    },
    "satisfaction_employee_nature_travail"      : {
        "description"        : "Satisfaction vis-à-vis de la nature du travail",
        "type"               : "int64",
        "categorie"          : "Satisfaction",
        "unite"              : "score (1-4)",
        "a_supprimer"        : False,
    },
    "satisfaction_employee_equipe"              : {
        "description"        : "Satisfaction vis-à-vis de l'équipe de travail",
        "type"               : "int64",
        "categorie"          : "Satisfaction",
        "unite"              : "score (1-4)",
        "a_supprimer"        : False,
    },
    "satisfaction_employee_equilibre_pro_perso": {
        "description"        : "Satisfaction vis-à-vis de l'équilibre vie professionnelle/personnelle",
        "type"               : "int64",
        "categorie"          : "Satisfaction",
        "unite"              : "score (1-4)",
        "a_supprimer"        : False,
    },

    # === ENGAGEMENT ET PARTICIPATION ===
    "nombre_participation_pee"                  : {
        "description"        : "Nombre de participations au Plan d'Épargne Entreprise",
        "type"               : "int64",
        "categorie"          : "Engagement",
        "unite"              : "nombre",
        "a_supprimer"        : False,
    },

    # === MANAGEMENT ===

    "annes_sous_responsable_actuel"             : {
        "description"        : "Années passées sous le responsable actuel",
        "type"               : "int64",
        "categorie"          : "Management",
        "unite"              : "années",
        "a_supprimer"        : False,
    }


}

# ❇️  Conexión y Carga

In [137]:
# Cargar datos desde la vista de ingeniería de la Fase 1
query       = "SELECT * FROM v_features_engineering"
df_featured = pd.read_sql(query, engine)
df_featured


,emp_id,id_employee,age,genre,statut_marital,revenu_mensuel,departement,poste,nombre_experiences_precedentes,nombre_heures_travailless,...,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel,fe1_ratio_stagnation,fe2_stabilite_manager,fe3_indice_job_hopping,fe4_anciennete_relative,fe5_satisfaction_globale,fe6_risque_overwork,fe7_penibilite_trajet,fe8_valeur_experience
0,1,1,41,F,Célibataire,5993,Commercial,Cadre Commercial,8,80,...,0,5,0.0000,1.0000,0.8889,0.2609,2.00,0.5000,1,665.89
1,10,10,59,F,Marié(e),2670,Consulting,Consultant,4,80,...,0,0,0.0000,0.0000,2.4000,0.0244,1.75,0.3333,3,205.38
2,100,100,35,M,Célibataire,4312,Commercial,Cadre Commercial,0,80,...,2,8,0.1250,0.5714,16.0000,0.8824,2.25,0.0000,0,253.65
3,1001,1001,27,F,Marié(e),2811,Consulting,Consultant,9,80,...,2,2,0.6667,0.6667,0.4000,0.2222,2.50,0.0000,0,562.20
4,1002,1002,45,M,Marié(e),3633,Consulting,Consultant,1,80,...,0,8,0.0000,0.8889,4.5000,0.3333,2.75,0.2500,23,363.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,995,995,33,F,Célibataire,3452,Consulting,Assistant de Direction,3,80,...,0,2,0.0000,0.6667,1.2500,0.2000,2.50,0.0000,0,575.33
1466,996,996,45,F,Divorcé(e),2270,Consulting,Consultant,3,80,...,0,2,0.0000,0.5000,2.0000,0.1852,3.75,0.0000,0,252.22
1467,997,997,50,M,Divorcé(e),17399,Consulting,Directeur Technique,9,80,...,1,3,0.1667,0.6000,3.2000,0.1563,3.25,0.0000,0,527.24
1468,998,998,33,F,Marié(e),5488,Consulting,Manager,1,80,...,1,2,0.1429,0.3333,3.0000,0.4000,2.25,0.2500,1,784.00


In [140]:
inspect_table_columns("v_features_engineering", engine)


✅ Estructura de la tabla: v_features_engineering


,column_name,data_type,is_nullable
0,emp_id,character varying,YES
1,id_employee,bigint,YES
2,age,bigint,YES
3,genre,text,YES
4,statut_marital,text,YES
5,revenu_mensuel,bigint,YES
6,departement,text,YES
7,poste,text,YES
8,nombre_experiences_precedentes,bigint,YES
9,nombre_heures_travailless,bigint,YES


## ⬇️ Use Classe auxiliare pour analyse (DataCleaner)

In [141]:
cleaner = DataCleaner( df = df_featured, verbose = True  )

# Definición de la Estrategia de Selección

## ✂️ Step 4. SPLIT X Y 

In [142]:
# --- CONFIGURACIÓN DE COLUMNAS (Basado en tu DataCleaner) ---
# Identificamos las que vamos a sacrificar

FEATURES_TO_REMOVE = COLS_IDENTIFIANTS + COLS_CONSTANT + COLS_MISSING_EXCESIF + COLS_REDONDANTS
cleaner.FEATURES_TO_REMOVE = FEATURES_TO_REMOVE

# 1. Limpieza Inicial (Drop de columnas basura y separación de Target)
X_prepared, y = cleaner.pipeline_preparation_initiale(
    df           = cleaner.df,
    target       = FEATURE_TARGET,
    cols_to_drop = FEATURES_TO_REMOVE
)

cleaner.COLS_TO_LOG                   = COLS_TO_LOG
cleaner.COLS_TO_ROBUST                = COLS_TO_ROBUST
cleaner.COLS_STANDARD                 = COLS_STANDARD
cleaner.COLS_TO_LOG                   = COLS_TO_LOG
cleaner.COLS_ONE_HOT_ENCODING         = COLS_ONE_HOT_ENCODING
cleaner.COLS_TARGET_ADVANCED_ENCODING = COLS_TARGET_ADVANCED_ENCODING

# 2. Identificación Dinámica (El corazón del Pipeline)
columnas_vivas = set(X_prepared.columns)

info_final = {
    'num_log':    [c for c in cleaner.COLS_TO_LOG    if c in columnas_vivas],
    'num_robust': [c for c in cleaner.COLS_TO_ROBUST if c in columnas_vivas],
    'num_std':    [c for c in cleaner.COLS_STANDARD  if c in columnas_vivas],
    'cat_ohe':    [c for c in cleaner.COLS_ONE_HOT_ENCODING if c in columnas_vivas],
    'cat_target': [c for c in cleaner.COLS_TARGET_ADVANCED_ENCODING if c in columnas_vivas],
    'indic_nan':  [c for c in X_prepared.columns if X_prepared[c].isnull().any()]
}


--- Nettoyage des colonnes ---
ℹ️ Colonne ignorée (déjà absente) : id
🗑️ Suppression de : id_employee
ℹ️ Colonne ignorée (déjà absente) : eval_number
ℹ️ Colonne ignorée (déjà absente) : code_sondage
🗑️ Suppression de : nombre_heures_travailless
🗑️ Suppression de : nombre_employee_sous_responsabilite
🗑️ Suppression de : ayant_enfants
------------------------------------------------------------
🎯 CIBLE (y)      : target_attrition
🧬 FEATURES (X)   : ['emp_id', 'age', 'genre', 'statut_marital', 'revenu_mensuel', 'departement', 'poste', 'nombre_experiences_precedentes', 'annee_experience_totale', 'annees_dans_l_entreprise', 'annees_dans_le_poste_actuel', 'satisfaction_employee_environnement', 'satisfaction_employee_nature_travail', 'satisfaction_employee_equipe', 'satisfaction_employee_equilibre_pro_perso', 'note_evaluation_actuelle', 'note_evaluation_precedente', 'niveau_hierarchique_poste', 'heure_supplementaires', 'augementation_salaire_precedente', 'nombre_participation_pee', 'nb_forma

In [143]:
X_prepared

,emp_id,age,genre,statut_marital,revenu_mensuel,departement,poste,nombre_experiences_precedentes,annee_experience_totale,annees_dans_l_entreprise,...,annees_depuis_la_derniere_promotion,annes_sous_responsable_actuel,fe1_ratio_stagnation,fe2_stabilite_manager,fe3_indice_job_hopping,fe4_anciennete_relative,fe5_satisfaction_globale,fe6_risque_overwork,fe7_penibilite_trajet,fe8_valeur_experience
0,1,41,F,Célibataire,5993,Commercial,Cadre Commercial,8,8,6,...,0,5,0.0000,1.0000,0.8889,0.2609,2.00,0.5000,1,665.89
1,10,59,F,Marié(e),2670,Consulting,Consultant,4,12,1,...,0,0,0.0000,0.0000,2.4000,0.0244,1.75,0.3333,3,205.38
2,100,35,M,Célibataire,4312,Commercial,Cadre Commercial,0,16,15,...,2,8,0.1250,0.5714,16.0000,0.8824,2.25,0.0000,0,253.65
3,1001,27,F,Marié(e),2811,Consulting,Consultant,9,4,2,...,2,2,0.6667,0.6667,0.4000,0.2222,2.50,0.0000,0,562.20
4,1002,45,M,Marié(e),3633,Consulting,Consultant,1,9,9,...,0,8,0.0000,0.8889,4.5000,0.3333,2.75,0.2500,23,363.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,995,33,F,Célibataire,3452,Consulting,Assistant de Direction,3,5,3,...,0,2,0.0000,0.6667,1.2500,0.2000,2.50,0.0000,0,575.33
1466,996,45,F,Divorcé(e),2270,Consulting,Consultant,3,8,5,...,0,2,0.0000,0.5000,2.0000,0.1852,3.75,0.0000,0,252.22
1467,997,50,M,Divorcé(e),17399,Consulting,Directeur Technique,9,32,5,...,1,3,0.1667,0.6000,3.2000,0.1563,3.25,0.0000,0,527.24
1468,998,33,F,Marié(e),5488,Consulting,Manager,1,6,6,...,1,2,0.1429,0.3333,3.0000,0.4000,2.25,0.2500,1,784.00


# Construcción del Pipeline de Scikit-Learn

## ⬇️ LOAD Transformation Functions

In [144]:
# ------------------------------------------------------------------------------
def log_transform(df_x):
    """Applique log(1+x) avec gestion des valeurs négatives."""
    df_x             = df_x.copy()
    for col in df_x.columns:
        if df_x[col].min() < 0:
            offset   = abs(df_x[col].min()) + 1
            df_x[col] = np.log1p(df_x[col] + offset)
        else:
            df_x[col] = np.log1p(df_x[col])
    return df_x

# ------------------------------------------------------------------------------

def winsorize_transform(df_x, lower=0.01, upper=0.99):
    """Limite les valeurs extrêmes aux percentiles définis."""
    df_x             = df_x.copy()
    for col in df_x.columns:
        q_low        = df_x[col].quantile(lower)
        q_high       = df_x[col].quantile(upper)
        df_x[col]    = df_x[col].clip(lower=q_low, upper=q_high)
    return df_x

# On crée des fonctions simples qui appellent les originales avec les bons paramètres
# Cela remplace les lambdas qui bloquent le joblib.dump
def wrapper_winsorize(df_x):
    return winsorize_transform(df_x, lower=0.01, upper=0.99)

# ------------------------------------------------------------------------------

def create_missing_indicators(df_x, cols_to_flag):
    """Crée des drapeaux binaires pour les valeurs manquantes importantes."""
    df_x             = df_x.copy()
    for col in cols_to_flag:
        if col in df_x.columns:
            df_x[f'{col}_Nan'] = df_x[col].isna().astype(int)
    return df_x

# On crée des fonctions simples qui appellent les originales avec les bons paramètres
# Cela remplace les lambdas qui bloquent le joblib.dump
def wrapper_missing_indicators(df_x, cols_to_flag):
    return create_missing_indicators(df_x, cols_to_flag)

## ⬇️ LOAD Create features

In [145]:
def create_features(df_x):
    """Génère de nouvelles variables métier (Feature Engineering)."""
    df_x             = df_x.copy()

    # TODO plus tard

    return df_x

## ⬇️ LOAD Funtion Create Pre-Pipeline

In [146]:
def build_preprocessing_pipeline(df_x, v_y=None, info=None):
    """
    Assemble les briques de transformation de manière dynamique.
    
    Paramètres:
    -----------
    df_x : DataFrame d'entrée (Features)
    v_y  : Series cible (nécessaire pour le TargetEncoder)
    info : Dictionnaire généré par identifier_colonnes contenant la répartition des listes.
    """

    # Sécurité : Si info n'est pas fourni, on tente de le générer (mais il vaut mieux le passer)
    if info is None:
        # On suppose ici qu'une version simplifiée existe ou on lève une erreur
        raise ValueError("Le paramètre 'info' (mapping des colonnes) est obligatoire.")

    # --- 1.1 Sous-pipeline : ASYMÉTRIE CRITIQUE (Log + Winsor + Scaling) ---
    pipe_num_log = Pipeline(steps=[
        ('log',     FunctionTransformer(log_transform        , validate=False)),
        ('winsor',  FunctionTransformer(winsorize_transform  , validate=False, kw_args={'lower': 0.01, 'upper': 0.99})),
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])

    # --- 1.2 Sous-pipeline : ROBUSTE (Outliers, CV > 2) ---
    pipe_num_robust = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  RobustScaler())
    ])

    # --- 1.3 Sous-pipeline : STANDARD (RAS) ---
    pipe_num_std = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ])

    # --- 1.4 Sous-pipeline : CATÉGORIEL (One-Hot) ---
    pipe_cat_ohe = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant'    , fill_value='NULL')),
        ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    # --------------------------------------------------------------------------
    # 2. CONSTRUCTION DU DISPATCH
    # --------------------------------------------------------------------------
    dispatch = []

    # On ajoute les branches uniquement si les listes ne sont pas vides
    # Cela évite les erreurs de Scikit-Learn sur des listes vides
    if info['num_log']:    dispatch.append(('n_log',    pipe_num_log,    info['num_log']))
    if info['num_robust']: dispatch.append(('n_robust', pipe_num_robust, info['num_robust']))
    if info['num_std']:    dispatch.append(('n_std',    pipe_num_std,    info['num_std']))
    if info['cat_ohe']:    dispatch.append(('c_ohe',    pipe_cat_ohe,    info['cat_ohe']))

    if v_y is not None and info.get('cat_target'):
        pipe_target = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='NULL')),
            ('target',  TargetEncoder(smoothing=10, min_samples_leaf=20))
        ])
        dispatch.append(('c_target', pipe_target, info['cat_target']))

    preprocessor = ColumnTransformer(transformers=dispatch, remainder='drop')

    # --------------------------------------------------------------------------
    # 3. PIPELINE GLOBAL
    # --------------------------------------------------------------------------
    return Pipeline(steps=[
        ('nan_flags', FunctionTransformer(create_missing_indicators, validate=False, kw_args={'cols_to_flag': info['indic_nan']})),
        ('feat_eng' , FunctionTransformer(create_features, validate=False)),
        ('preproc'  , preprocessor)
    ])

In [147]:
# Aquí insertamos tu función build_preprocessing_pipeline(df_x, v_y, info)
# que ya tienes definida.

# Generamos el pipeline completo
preprocessing_pipe = build_preprocessing_pipeline(X_prepared, v_y=y, info=info_final)

# 1. Configuramos la salida a Pandas para no perder los nombres de las columnas
preprocessing_pipe.set_output(transform="pandas")

# 2. AJUSTAMOS Y TRANSFORMAMOS (Generación de la Matriz de Diseño)
# Aquí 'fit' aprende de los datos y 'transform' crea el DataFrame final
X_final_scaled_df = preprocessing_pipe.fit_transform(X_prepared, y)
X_final_scaled_df

,n_log__revenu_mensuel,n_log__annee_experience_totale,n_log__annees_dans_l_entreprise,n_log__annees_depuis_la_derniere_promotion,n_std__age,n_std__nombre_experiences_precedentes,n_std__annees_dans_le_poste_actuel,n_std__satisfaction_employee_environnement,n_std__note_evaluation_precedente,n_std__satisfaction_employee_nature_travail,...,c_ohe__poste_Tech Lead,c_ohe__domaine_etude_Autre,c_ohe__domaine_etude_Entrepreunariat,c_ohe__domaine_etude_Infra & Cloud,c_ohe__domaine_etude_Marketing,c_ohe__domaine_etude_Ressources Humaines,c_ohe__domaine_etude_Transformation Digitale,c_ohe__frequence_deplacement_Aucun,c_ohe__frequence_deplacement_Frequent,c_ohe__frequence_deplacement_Occasionnel
0,0.218262,-0.148970,0.178857,-0.974295,0.446350,2.125136,-0.063296,-0.660531,0.379672,1.153254,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1,-1.006363,0.389579,-1.480960,-0.974295,2.417384,0.523316,-1.167687,0.254625,1.785511,-1.567907,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,-0.280380,0.782464,1.274144,0.384858,-0.210661,-1.078504,2.421585,0.254625,-1.026167,-1.567907,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,-0.928424,-1.009810,-0.943749,0.384858,-1.086676,2.525591,-0.615492,0.254625,0.379672,-0.660853,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,-0.539907,0.005335,0.651425,-0.974295,0.884358,-0.678049,1.041095,1.169781,0.379672,-1.567907,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,-0.617312,-0.742792,-0.562592,-0.974295,-0.429664,0.122861,-0.615492,-0.660531,0.379672,1.153254,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1466,-1.252153,-0.148970,-0.025381,-0.974295,0.884358,0.122861,-0.339394,1.169781,0.379672,1.153254,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1467,1.832868,1.753887,-0.025381,-0.116765,1.431867,2.525591,-0.063296,1.169781,0.379672,1.153254,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1468,0.084918,-0.517031,0.178857,-0.116765,-0.429664,-0.678049,0.212802,0.254625,1.785511,-0.660853,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


# Visualización y Validación

In [148]:
# Verificamos las nuevas dimensiones tras el One-Hot Encoding
print(f"Dimensiones originales.....: {X_prepared.shape}")
print(f"Dimensiones tras Pipeline..: {X_final_scaled_df.shape}")

# Opcional: ver los nombres de las columnas generadas
#if hasattr(preprocessing_pipe.named_steps['preproc'], 'get_feature_names_out'):
#    cols_out = preprocessing_pipe.named_steps['preproc'].get_feature_names_out()
#    print(f"Total features finales: {len(cols_out)}")

Dimensiones originales.....: (1470, 36)
Dimensiones tras Pipeline..: (1470, 43)


# Verifier Equivalence

In [149]:
def verifier_equivalence(df_raw, df_final):
    print("📋 DIAGNOSTIC D'ÉQUIVALENCE")
    print("-" * 50)

    # 1. Verificación de Filas (Crucial)
    rows_match = len(df_raw) == len(df_final)
    print(f"✅ Coïncidence des lignes : {'OUI' if rows_match else 'NON'} ({len(df_raw)} vs {len(df_final)})")

    # 2. Verificación de NaNs
    nans_final = df_final.isna().sum().sum()
    print(f"✅ Absence de NaNs en X_final_df : {'OUI' if nans_final == 0 else 'NON'} (Trouvé: {nans_final})")

    # 3. Rastreo de columnas transformadas
    # Las columnas OHE suelen tener el prefijo 'c_ohe__' o similar
    ohe_cols = [c for c in df_final.columns if 'c_ohe' in c]
    std_cols = [c for c in df_final.columns if 'n_std' in c]
    log_cols = [c for c in df_final.columns if 'n_log' in c]
    rob_cols = [c for c in df_final.columns if 'n_robust' in c] # Añadimos robust por seguridad
    nan_cols = [c for c in df_final.columns if 'nan_flags' in c] # Flags de nulos si los hay

    print(f"ohe_cols.....: {ohe_cols}")
    print(f"log_cols.....: {log_cols}")
    print(f"rob_cols.....: {rob_cols}")
    print(f"std_cols.....: {std_cols}")
    print(f"nan_cols.....: {nan_cols}")

    # Suma de identificadas (sin contar nan_flags para no duplicar si se aplicó sobre numéricas)
    identificadas = set(ohe_cols + std_cols + log_cols + rob_cols)
    total_cols_final = df_final.shape[1]
    desconocidas = [c for c in df_final.columns if c not in identificadas and 'nan_flags' not in c]

    W_REPORT = 35
    print("\n" + "-" * 65)
    print(f"{'🔎 BILAN DE TRANSFORMATION DES COLONNES':^65}")
    print("-" * 65)
    print(f"  {'Colonnes OHE (Categorical)    ':<{W_REPORT}} : {len(ohe_cols):>4}")
    print(f"  {'Colonnes Log Transform        ':<{W_REPORT}} : {len(log_cols):>4}")
    print(f"  {'Colonnes Robust Scale         ':<{W_REPORT}} : {len(rob_cols):>4}")
    print(f"  {'Colonnes Standard Scale       ':<{W_REPORT}} : {len(std_cols):>4}")
    print(f"  {'Colonnes Missing Flags (Extra)':<{W_REPORT}} : {len(nan_cols):>4}")
    print("-" * 65)
    print(f"  {'TOTAL COLONNES IDENTIFIÉES    ':<{W_REPORT}} : {len(identificadas):>4}")
    print(f"  {'TOTAL COLONNES DANS X_FINAL   ':<{W_REPORT}} : {total_cols_final:>4}")

    diff = total_cols_final - len(identificadas) - len(nan_cols)
    if diff == 0:
        print("  ✅ CONCORDANCE PARFAITE : Toutes les colonnes sont tracées.")
    else:
        print(f"  ⚠️ ALERTE : {diff} colonnes non identifiées.")
        if desconocidas:
            print(f"  🧐 Colonnes 'fantômes' : {desconocidas[:5]}...")
    print("-" * 65)

    # 4. Prueba de correlación (Para variables que no cambiaron de nombre)
    # Si una variable solo se escaló (StandardScaler), su correlación con la original debe ser 1.0
    col_test = df_raw.select_dtypes(include=[np.number]).columns[0]
    # Buscamos su versión en el final (usualmente tiene un prefijo de la rama del transformer)
    col_final = [c for c in df_final.columns if col_test in c][0]

    corr = np.corrcoef(df_raw[col_test].fillna(0), df_final[col_final])[0, 1]
    print(f"✅ Intégrité Numérique ({col_test}) : Corrélaltion = {corr:.4f}")

    return rows_match and nans_final == 0



In [150]:
# 3. DIAGNÓSTICO
# Ahora que tienes el DataFrame, verificamos que la "fábrica" funciona bien
verifier_equivalence(X_prepared, X_final_scaled_df)

📋 DIAGNOSTIC D'ÉQUIVALENCE
--------------------------------------------------
✅ Coïncidence des lignes : OUI (1470 vs 1470)
✅ Absence de NaNs en X_final_df : OUI (Trouvé: 0)
ohe_cols.....: ['c_ohe__genre_F', 'c_ohe__genre_M', 'c_ohe__statut_marital_Célibataire', 'c_ohe__statut_marital_Divorcé(e)', 'c_ohe__statut_marital_Marié(e)', 'c_ohe__poste_Assistant de Direction', 'c_ohe__poste_Cadre Commercial', 'c_ohe__poste_Consultant', 'c_ohe__poste_Directeur Technique', 'c_ohe__poste_Manager', 'c_ohe__poste_Représentant Commercial', 'c_ohe__poste_Ressources Humaines', 'c_ohe__poste_Senior Manager', 'c_ohe__poste_Tech Lead', 'c_ohe__domaine_etude_Autre', 'c_ohe__domaine_etude_Entrepreunariat', 'c_ohe__domaine_etude_Infra & Cloud', 'c_ohe__domaine_etude_Marketing', 'c_ohe__domaine_etude_Ressources Humaines', 'c_ohe__domaine_etude_Transformation Digitale', 'c_ohe__frequence_deplacement_Aucun', 'c_ohe__frequence_deplacement_Frequent', 'c_ohe__frequence_deplacement_Occasionnel']
log_cols.....: ['n

np.True_

# 💾  Enregistrer Union de Features et Target

In [152]:
# 1. Unimos las features transformadas con la target original
# Usamos axis=1 para unir por columnas
data_output = pd.concat([X_final_scaled_df, y], axis=1)

print(f"✅ Matrice finale consolidée : {data_output.shape[0]} lignes x {data_output.shape[1]} colonnes")

✅ Matrice finale consolidée : 1470 lignes x 44 colonnes


## Enregistrement

In [159]:
# Definir la ruta profesional
output_path = PREPROCESED_DIR / "data_final.csv"

# Crear carpeta si no existe (Seguridad de sistema)
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Guardar
data_output.to_csv(output_path, index=False)
print(f"💾 Fichier sauvegardé avec succès : {output_path}")

💾 Fichier sauvegardé avec succès : ..\data\processed\data_final.csv


In [160]:

# Créer un pack complet (Version Robuste)
export_data = {
    'preprocessor' : preprocessing_pipe['preproc']       # LE COLUMNTRANSFORMER
}

# Sauvegarde
output_path = PREPROCESED_DIR / "0_BEST_COLUMNTRANSFORMER_PACK.joblib"
joblib.dump(export_data, output_path)

print(f"✅ COLUMNTRANSFORMER sauvegardé : {output_path}")

✅ COLUMNTRANSFORMER sauvegardé : ..\data\processed\0_BEST_COLUMNTRANSFORMER_PACK.joblib
